In [ ]:
# 1. Clone the official SketchEdit CVPR repository
!git clone https://github.com/zengxianyu/sketchedit.git

# 2. Install ControlNet, Diffusers, Gradio, and Optimization packages
!pip install -q diffusers accelerate controlnet_aux transformers gradio Pillow
!pip install -q xformers || true

# 3. Append the repository to the system path to allow Python module imports
import sys
sys.path.append('/content/sketchedit')

Cloning into 'sketchedit'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 318 (delta 117), reused 295 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (318/318), 28.35 MiB | 12.57 MiB/s, done.
Resolving deltas: 100% (117/117), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.3 MB/s eta 0:00:00


In [ ]:
# 1. Navigate into the repository
%cd /content/sketchedit

# 2. Grant execution permissions to the download scripts
!chmod +x download/* test_places.sh

# 3. Download the core model weights
!./download/download_model.sh

# Note: Watch the output of this cell to confirm the exact path
# where the Places checkpoint is saved (usually ./checkpoints/places.pth)

/content/sketchedit
--2026-04-19 17:00:41--  https://maildluteducn-my.sharepoint.com/:u:/g/personal/zengyu_mail_dlut_edu_cn/ETFFjNQcRzFMs2Dgq6tiVnIB3msKBXYMBOSWnVYEHgrlxQ?download=1
Resolving maildluteducn-my.sharepoint.com (maildluteducn-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to maildluteducn-my.sharepoint.com (maildluteducn-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/zengyu_mail_dlut_edu_cn/Documents/celeb.zip?ga=1 [following]
--2026-04-19 17:00:42--  https://maildluteducn-my.sharepoint.com/personal/zengyu_mail_dlut_edu_cn/Documents/celeb.zip?ga=1
Reusing existing connection to maildluteducn-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 27887718 (27M) [application/x-zip-compressed]
Saving to: ‘ETFFjNQcRzFMs2Dgq6tiVnIB3msKBXYMBOSWnVYEHgrlxQ?download=1’

ETFFjNQcRzFMs2Dgq6t 100%[===================>]  26.60M  10.4MB/s    in 2.6s    

In [ ]:
import os
import sys
import subprocess
import glob
import torch
import numpy as np
from PIL import Image, ImageDraw
import torchvision.transforms as T
import gradio as gr
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)
from controlnet_aux import HEDdetector

# ======================================================================
# Phase 1: ControlNet Setup (Text + Sketch)
# ======================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Base Generation Models on {device}...")

try:
    controlnet_model = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_scribble",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )
    controlnet_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet_model,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    )
    controlnet_pipe.scheduler = UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config)
    if device == "cuda":
        controlnet_pipe = controlnet_pipe.to(device)
        controlnet_pipe.enable_xformers_memory_efficient_attention()

    scribble_processor = HEDdetector.from_pretrained("lllyasviel/Annotators")
except Exception as e:
    print(f"ControlNet Error: {e}")

# ======================================================================
# Phase 2: SketchEdit Wrapper (Native Subprocess Execution)
# ======================================================================
class SketchEditPipeline:
    def __init__(self):
        self.base_temp = '/content/sketchedit/temp_data'

        os.makedirs(os.path.join(self.base_temp, 'images'), exist_ok=True)
        os.makedirs(os.path.join(self.base_temp, 'sketches'), exist_ok=True)

        self.list_file = os.path.join(self.base_temp, 'image_list.txt')
        with open(self.list_file, 'w') as f:
            f.write('input\n')

        print("SketchEdit Subprocess Pipeline is ONLINE.")

    def edit(self, base_image_pil, partial_sketch_pil):
        img_path = os.path.join(self.base_temp, 'images', 'input.png')
        skt_path = os.path.join(self.base_temp, 'sketches', 'input.png')

        base_image_pil.convert('RGB').resize((256, 256), Image.LANCZOS).save(img_path)
        partial_sketch_pil.convert('L').resize((256, 256), Image.NEAREST).save(skt_path)

        out_dir = os.path.join(self.base_temp, 'outputs')
        if os.path.exists(out_dir):
            import shutil
            shutil.rmtree(out_dir)
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            sys.executable, "test.py",
            "--name", "places",
            "--model", "editline2",
            "--netG", "deepfillc2",
            "--dataset_mode", "testimage",
            "--image_dirs", os.path.join(self.base_temp, 'images'),
            "--mask_dirs", os.path.join(self.base_temp, 'sketches'),
            "--image_lists", self.list_file,
            "--output_dir", out_dir,
            "--checkpoints_dir", "/content/sketchedit/checkpoints",
            "--image_postfix", ".png",
            "--mask_postfix", ".png",
            "--gpu_ids", "0" if torch.cuda.is_available() else "-1",
            "--pool_type", "max",
            "--batchSize", "1",
            "--nThreads", "0",
            "--joint_train_inp",
            "--preprocess", "resize_and_crop",
            "--load_size", "256",
            "--crop_size", "256"
        ]

        result = subprocess.run(cmd, cwd="/content/sketchedit", capture_output=True, text=True)

        if result.returncode != 0:
            raise RuntimeError(f"Subprocess Failed. Error Log:\n{result.stderr[-1000:]}")

        out_files = glob.glob(os.path.join(out_dir, '**', '*.*'), recursive=True)
        out_files = [f for f in out_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if not out_files:
            raise RuntimeError(f"Model ran but no image was saved.\nSTDOUT:\n{result.stdout[-1000:]}\nSTDERR:\n{result.stderr[-1000:]}")

        synthesized_files = [f for f in out_files if 'synthesized' in f.lower() or 'comp' in f.lower()]
        latest_out = synthesized_files[0] if synthesized_files else sorted(out_files, key=os.path.getmtime)[-1]

        out_pil = Image.open(latest_out).convert('RGB')
        return out_pil

sketch_edit_pipe = SketchEditPipeline()

# ======================================================================
# Utilities
# ======================================================================
def extract_drawing(data):
    if data is None or "background" not in data or "layers" not in data:
        return None, None
    bg = data["background"].convert("RGB")
    stroke = Image.new("RGBA", bg.size, (0,0,0,0))
    for layer in data["layers"]:
        stroke.alpha_composite(layer.convert("RGBA"))

    # ---------------------------------------------------------
    # SKETCHEDIT FIX: DeepFill-v2 treats White (255) as the area
    # to modify and Black (0) as the area to preserve.
    # ---------------------------------------------------------
    sketch = Image.new("L", bg.size, 0) # Create solid BLACK background

    # Extract the alpha channel (where the user actually drew)
    alpha = stroke.split()[3]
    if alpha.getextrema()[1] > 0:
        white_stroke = Image.new("L", bg.size, 255) # Create WHITE stroke
        sketch.paste(white_stroke, mask=alpha)

    return bg, sketch

def make_text_image(msg: str, size=(512, 512)):
    img = Image.new("RGB", size, color=(255, 220, 180))
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), msg, fill=(0, 0, 0))
    return img

# ======================================================================
# Gradio UI
# ======================================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("<h1 style='text-align: center;'>🎨 Scene Modeler: ControlNet ➡️ SketchEdit</h1>")
    gr.Markdown("<p style='text-align: center;'>Synthesize a base scene, then sketch directly over it for mask-free editing.</p>")

    base_image_state = gr.State(None)

    with gr.Tabs():
        # --- PHASE 1: Text + Sketch ---
        with gr.TabItem("Phase 1: Concept Synthesis"):
            with gr.Row(equal_height=True):
                with gr.Column(scale=1):
                    raw_sketch_input = gr.ImageEditor(label="1. Draw Concept Sketch", type="pil", interactive=True, height=512)
                with gr.Column(scale=1):
                    base_output = gr.Image(label="2. Generated Concept", type="pil", height=512, interactive=False)

            with gr.Row():
                prompt_input = gr.Textbox(label="Scene Prompt", value="A highly detailed landscape masterpiece, lush mountains, clear lake, cinematic lighting", scale=4)
                gen_btn = gr.Button("🎨 Generate Concept", variant="primary", scale=1)

            with gr.Row():
                send_to_edit_btn = gr.Button("🔒 Lock Image & Send to Phase 2 ➡️", variant="secondary")

        # --- PHASE 2: Sketch-Only Editing ---
        with gr.TabItem("Phase 2: Mask-Free SketchEdit"):
            gr.Markdown("<h3 style='text-align: center;'>Predicting modification regions based on user sketch inputs.</h3>")

            with gr.Row(equal_height=True):
                with gr.Column(scale=1):
                    edit_sketch_input = gr.ImageEditor(label="3. Draw Edits Over Base Image", type="pil", interactive=True, height=512)
                with gr.Column(scale=1):
                    final_output = gr.Image(label="4. Final Manipulated Image", type="pil", height=512, interactive=False)

            with gr.Row():
                edit_btn = gr.Button("✨ Synthesize Edit", variant="primary")

    # --- Logic Routing ---
    def run_controlnet(gr_input, prompt):
        bg, sketch = extract_drawing(gr_input)
        if bg is None: return make_text_image("Please draw a sketch first.")

        scribble_np = scribble_processor(sketch.convert("RGB"), scribble=True)
        img = controlnet_pipe(
            prompt=prompt,
            negative_prompt="low quality, messy, faces, people, text, ugly",
            image=scribble_np,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]
        return img

    def bridge_handoff(generated_img):
        return generated_img, generated_img

    def run_sketchedit(gr_input, base_img_state):
        bg, partial_sketch = extract_drawing(gr_input)
        try:
            result = sketch_edit_pipe.edit(bg, partial_sketch)
            return result.resize((512, 512), Image.LANCZOS)
        except Exception as e:
            return make_text_image(f"Error: {str(e)}")

    gen_btn.click(fn=run_controlnet, inputs=[raw_sketch_input, prompt_input], outputs=base_output)
    send_to_edit_btn.click(fn=bridge_handoff, inputs=base_output, outputs=[edit_sketch_input, base_image_state])
    edit_btn.click(fn=run_sketchedit, inputs=[edit_sketch_input, base_image_state], outputs=final_output)

demo.launch(debug=True, share=True)

Booting Base Generation Models on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://740437d4fb7be4caf1.gradio.live
